# 生成式 AI 應用開發｜第 12 週：多模態應用、Vision API 與圖片理解（學生版）

本週承接第 6～8 週的 Structured Outputs 與 Streamlit、第 9～11 週的輸入驗證與可靠性觀念，把輸入從文字／文件延伸到圖片。正式主線使用 **OpenAI Responses API + `input_image`**，成果是可在 VS Code 執行的圖片理解工具。

## 學習目標

完成本週後，你應能：

1. 解釋圖片如何以 URL、Base64 data URL 或 file ID 進入模型。
2. 驗證圖片格式、大小、尺寸與 MIME，而不是只相信副檔名。
3. 使用 Responses API 組合 `input_text` 與 `input_image`。
4. 設計圖片描述、圖片問答、截圖理解與文件抽取 prompt。
5. 用 JSON Schema 接收可由程式讀取的視覺文件欄位。
6. 處理拒答、未完成、空輸出、解析失敗、成本與隱私風險。

## 先備知識與安全紅線

- 已理解環境變數、例外處理、Responses API、JSON Schema 與 Streamlit 表單。
- 圖片會傳送到外部 API；課堂只用自己製作或明確授權、且不含敏感資料的圖片。
- 不上傳身分證、成績、病歷、醫療影像、金融資料、公司機密、未授權人像或 CAPTCHA。
- Vision 可能看錯小字、數字、物件數量、圖表與空間關係；重要結果一定回看原圖。

## 3 小時建議流程

| 時間 | 主題 | 產出 |
|---:|---|---|
| 0–35 分 | 多模態心智模型、格式、detail、成本與限制 | 看懂圖片輸入資料流 |
| 35–85 分 | Base64、Responses API、錯誤分流 | 完成最小圖片問答 |
| 85–130 分 | Prompt 與 Structured Outputs | 完成收據／表單抽取 |
| 130–170 分 | 四組練習與測試 | 完成 helper 與基本評估 |
| 170–180 分 | Streamlit 專案說明與第 13 週銜接 | 能啟動圖片理解 App |

In [ ]:
# Colab 或全新環境執行一次；若已在課堂固定環境中可略過。
# 付費 API cell 預設不執行，避免「全部執行」直接產生成本。
!pip install openai==2.49.0 python-dotenv==1.2.3 pillow==12.3.0

In [ ]:
from __future__ import annotations

import base64
from io import BytesIO
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from PIL import Image, ImageDraw


MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
MAX_FILE_BYTES = 8 * 1024 * 1024
MAX_IMAGE_PIXELS = 20_000_000
RUN_PAID_API = False
print(f"模型：{MODEL}｜付費 API：{RUN_PAID_API}")

In [ ]:
def get_api_key() -> str | None:
    """優先讀取 Colab Secrets，再讀取本機 `.env`；絕不在輸出中顯示金鑰。"""
    try:
        from google.colab import userdata

        return userdata.get("OPENAI_API_KEY")
    except Exception:
        load_dotenv()
        return os.getenv("OPENAI_API_KEY")


def create_client() -> OpenAI:
    """在真正呼叫 API 前檢查金鑰，提供較容易排錯的訊息。"""
    api_key = get_api_key()
    if not api_key:
        raise RuntimeError("找不到 OPENAI_API_KEY，請設定 Colab Secrets 或 `.env`。")
    return OpenAI(api_key=api_key)

## 1. 多模態心智模型

一個請求可以同時包含多種 content：文字任務使用 `input_text`，圖片使用 `input_image`。本週 App 選擇 **Base64 data URL**，因為 Streamlit 上傳器已經給我們 bytes，不必先把圖片公開放到網路。

資料流：`圖片 bytes → 本機驗證 → Base64 data URL → input_image → 模型 → 文字或 JSON → 人工核對`。

In [ ]:
# 建立完全虛構的測試收據，避免課堂範例含有真實個資或交易資料。
DEMO_IMAGE_PATH = Path("demo_receipt.png")
image = Image.new("RGB", (900, 1000), "white")
draw = ImageDraw.Draw(image)
demo_lines = [
    "DEMO CAFE - CLASSROOM FIXTURE",
    "Date: 2026-09-16",
    "Coffee       2 x 80       160",
    "Sandwich     1 x 95        95",
    "Notebook     1 x 45        45",
    "TOTAL TWD                 300",
    "This is fictional test data.",
]
for index, line in enumerate(demo_lines):
    draw.text((70, 80 + index * 110), line, fill="black", font_size=30)
image.save(DEMO_IMAGE_PATH, format="PNG")
print(f"已建立：{DEMO_IMAGE_PATH.resolve()}")

In [ ]:
# 用 context manager 關閉檔案 handle；Windows 才能在後續清理暫存檔。
with Image.open(DEMO_IMAGE_PATH) as opened_image:
    demo_image = opened_image.copy()
    demo_format = opened_image.format
display(demo_image)
print("尺寸：", demo_image.size, "格式：", demo_format)

## 2. 支援格式、detail 與成本

OpenAI 官方文件列出 PNG、JPEG、WEBP 與非動態 GIF。圖片會換算成輸入 token；多張圖、尺寸與細節等級都可能提高成本。

- `low`：適合粗略內容理解。
- `high`：適合一般高細節理解；文件小字仍可能辨識錯誤。
- `auto`：交由模型選擇預處理方式。

本教材不宣稱 detail 與成本固定成正比；請依模型文件與實際 usage 檢查。

In [ ]:
def detect_image_mime(file_bytes: bytes) -> str:
    """依檔案 signature 判斷格式，不只相信副檔名。"""
    if file_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
        return "image/png"
    if file_bytes.startswith(b"\xff\xd8\xff"):
        return "image/jpeg"
    if file_bytes.startswith((b"GIF87a", b"GIF89a")):
        return "image/gif"
    if len(file_bytes) >= 12 and file_bytes.startswith(b"RIFF") and file_bytes[8:12] == b"WEBP":
        return "image/webp"
    raise ValueError("不是支援的 PNG、JPEG、WEBP 或 GIF。")


demo_bytes = DEMO_IMAGE_PATH.read_bytes()
print(detect_image_mime(demo_bytes), len(demo_bytes), "bytes")

In [ ]:
def image_to_data_url(file_bytes: bytes, mime_type: str) -> str:
    """把二進位圖片轉成 API 可接收的 data URL。"""
    encoded = base64.b64encode(file_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


demo_data_url = image_to_data_url(demo_bytes, detect_image_mime(demo_bytes))
print(demo_data_url[:50] + "...")
print("data URL 字元數：", len(demo_data_url))

## 3. Responses API 的圖片輸入

`input` 是訊息陣列；同一個 user message 的 `content` 同時放任務文字與圖片。回傳可先讀 SDK 的 `output_text`，但正式 App 還要檢查 refusal 與 `status`，不能假設每次都有完整文字。

In [ ]:
def get_refusal_reason(response) -> str | None:
    """拒答內容不一定出現在 output_text，因此先掃描 output/content。"""
    for output_item in getattr(response, "output", []) or []:
        content = output_item.get("content", []) if isinstance(output_item, dict) else getattr(output_item, "content", [])
        for content_item in content or []:
            refusal = content_item.get("refusal") if isinstance(content_item, dict) else getattr(content_item, "refusal", None)
            if refusal:
                return str(refusal)
    return None


def require_completed_response(response) -> str:
    """只接受已完成且有文字的回應，避免把部分輸出當成正式答案。"""
    refusal = get_refusal_reason(response)
    if refusal:
        raise RuntimeError(f"AI 拒絕處理圖片：{refusal}")
    status = getattr(response, "status", None)
    if status == "incomplete":
        raise RuntimeError("AI 回應未完成，請縮小任務或更換合適測試圖片。")
    if status not in (None, "completed"):
        raise RuntimeError(f"AI 回應未成功（狀態：{status}）。")
    output_text = (getattr(response, "output_text", None) or "").strip()
    if not output_text:
        raise RuntimeError("AI 沒有回傳文字結果。")
    return output_text

In [ ]:
def analyze_image_bytes(
    file_bytes: bytes,
    prompt: str,
    *,
    detail: str = "auto",
    model: str = MODEL,
) -> str:
    """將圖片與問題一起送入 Responses API，回傳完整文字答案。"""
    if detail not in {"low", "high", "auto"}:
        raise ValueError("detail 只接受 low、high 或 auto。")
    mime_type = detect_image_mime(file_bytes)
    response = create_client().responses.create(
        model=model,
        instructions="使用繁體中文；區分直接可見證據與不確定推論；無法確認時不可猜測。",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": image_to_data_url(file_bytes, mime_type),
                    "detail": detail,
                },
            ],
        }],
        # 不保存供之後 API 取回，不等於圖片沒有傳送或處理。
        store=False,
    )
    return require_completed_response(response)

In [ ]:
if RUN_PAID_API:
    answer = analyze_image_bytes(
        demo_bytes,
        "請列出圖片中確實可見的文字與總額；看不清楚的欄位不要猜測。",
        detail="high",
    )
    print(answer)
else:
    print("已略過付費 Vision API。確認金鑰、圖片與費用設定後，再把 RUN_PAID_API 改成 True。")

## 4. 圖片 prompt：把「證據」與「推論」分開

只問「這張圖是什麼？」容易得到過度概括的答案。較可靠的 prompt 會指定：任務、輸出順序、不可猜測項目、看不清楚時的回覆方式，以及是否需要 JSON。

In [ ]:
PROMPTS = {
    "圖片描述": "先列直接可見的物件、人物、文字與場景，再把不確定解讀標成『可能的推論』。",
    "截圖檢查": "依序整理可見文字、目前狀態、可能問題與使用者可自行確認的下一步；不要假裝已操作畫面。",
    "收據抽取": "只抽取確實可見的商家、日期、幣別、總額與品項；模糊欄位使用 null 並說明。",
}
for name, prompt in PROMPTS.items():
    print(f"{name}：{prompt}")

## 練習 A：完成上傳圖片驗證

在呼叫 API 前檢查空檔、8 MB、格式、尺寸、像素量與動態 GIF。注意：上傳器的 `type=` 只提供便利性篩選，不是安全保證。

In [ ]:
def validate_upload(file_bytes: bytes, filename: str) -> dict:
    """檢查課堂上傳圖片，回傳檔名、格式、尺寸與 bytes 數。"""
    # TODO 1：檢查空檔與 8 MB 上限。
    # TODO 2：呼叫 detect_image_mime()，不要只相信副檔名。
    # TODO 3：用 Pillow 讀取尺寸，拒絕動態 GIF 與超過像素上限的圖片。
    return {
        "filename": filename,
        "mime_type": "image/png",
        "width": 0,
        "height": 0,
        "bytes": len(file_bytes),
    }


In [ ]:
# 先用虛構收據做 smoke test；學生版 scaffold 可先執行，但尺寸會在完成練習後才正確。
upload_info = validate_upload(demo_bytes, "demo_receipt.png")
assert upload_info["filename"] == "demo_receipt.png"
assert upload_info["bytes"] == len(demo_bytes)
print(upload_info)

## 5. detail 是需求選擇，不是品質開關

文件與小字通常需要較高細節；粗略分類可先用 low；不確定時用 auto。更高細節仍不保證 OCR、計數或位置判斷正確，而且模型支援的 detail 值可能不同。

## 練習 B：建立 detail 選擇規則

讓 helper 依任務文字回傳 `low`、`high` 或 `auto`。規則的目的不是取代模型文件，而是讓 App 的成本／品質選擇可以被測試與說明。

In [ ]:
def choose_detail(task: str) -> str:
    """依任務內容回傳 low、high 或 auto。"""
    # TODO：文件、小字、表單或截圖使用 high；粗略分類使用 low；其他使用 auto。
    return "auto"


In [ ]:
assert choose_detail("讀取收據小字") in {"high", "auto"}
assert choose_detail("粗略分類照片") in {"low", "auto"}
assert choose_detail("一般圖片問答") == "auto"
print("detail 規則 smoke test 通過")

## 6. 圖片 + Structured Outputs

當 UI 必須讀取日期、金額與品項時，不能只要求「請回 JSON」。應提供 JSON Schema、`strict=True`、所有必要欄位與 `additionalProperties=False`。無法確認的值用 `null`，不要逼模型猜答案。

In [ ]:
RECEIPT_SCHEMA = {
    "type": "object",
    "properties": {
        "merchant": {"type": ["string", "null"]},
        "date": {"type": ["string", "null"]},
        "currency": {"type": ["string", "null"]},
        "total": {"type": ["number", "null"]},
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "quantity": {"type": ["number", "null"]},
                    "amount": {"type": ["number", "null"]},
                },
                "required": ["name", "quantity", "amount"],
                "additionalProperties": False,
            },
        },
        "warnings": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["merchant", "date", "currency", "total", "items", "warnings"],
    "additionalProperties": False,
}
print(json.dumps(RECEIPT_SCHEMA, ensure_ascii=False, indent=2)[:600] + "...")

In [ ]:
def extract_receipt(file_bytes: bytes, *, model: str = MODEL) -> dict:
    """使用相同圖片輸入，加上 JSON Schema 取得結構化文件資料。"""
    mime_type = detect_image_mime(file_bytes)
    response = create_client().responses.create(
        model=model,
        instructions="只抽取圖片中可見資料；模糊值使用 null；不得補造。",
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": PROMPTS["收據抽取"]},
                {
                    "type": "input_image",
                    "image_url": image_to_data_url(file_bytes, mime_type),
                    "detail": "high",
                },
            ],
        }],
        text={"format": {
            "type": "json_schema",
            "name": "receipt_result",
            "schema": RECEIPT_SCHEMA,
            "strict": True,
        }},
        store=False,
    )
    output_text = require_completed_response(response)
    try:
        return json.loads(output_text)
    except json.JSONDecodeError as exc:
        raise RuntimeError("模型回傳內容無法解析成收據 JSON。") from exc

In [ ]:
if RUN_PAID_API:
    receipt = extract_receipt(demo_bytes)
    print(json.dumps(receipt, ensure_ascii=False, indent=2))
else:
    print("已略過付費抽取；可先閱讀 schema，並完成下方離線資料整理練習。")

## 練習 C：正規化抽取結果

外部回應進入 UI 前仍要整理資料結構。缺少清單時改成空 list；缺少單值時保留 `None`，不可偷偷用假資料補齊。

In [ ]:
def normalize_receipt(result: dict) -> dict:
    """把模型結果整理成 UI 可安全讀取的固定結構。"""
    # TODO：保留 merchant/date/currency/total，並把缺少的 items/warnings 轉成空 list。
    return {
        "merchant": None,
        "date": None,
        "currency": None,
        "total": None,
        "items": [],
        "warnings": ["尚未完成正規化函式"],
    }


In [ ]:
fake_receipt = {
    "merchant": "DEMO CAFE",
    "date": "2026-09-16",
    "currency": "TWD",
    "total": 300,
    "items": None,
}
normalized = normalize_receipt(fake_receipt)
assert isinstance(normalized["items"], list)
assert isinstance(normalized["warnings"], list)
print(normalized)

## 7. 基本評估：規則能檢查格式，不能證明看懂

Vision 評估至少要包含一組清楚圖片、一組小字／模糊圖片與一組圖片沒有答案的問題。下方規則只檢查回答是否提到證據與不確定性；真正品質仍需人工對照原圖與標準答案。

## 練習 D：建立可解釋的基本檢查

用簡單關鍵詞檢查答案是否提到圖片證據、是否標示不確定性，並固定提醒人工覆核。這種規則只能找出部分格式問題，不能證明答案正確。

In [ ]:
def evaluate_vision_answer(answer: str) -> dict:
    """檢查回答是否為空、是否提到圖片證據，以及是否標示不確定性。"""
    # TODO：以關鍵詞完成三項規則檢查，並固定標示 needs_human_review=True。
    return {
        "not_empty": bool(answer.strip()),
        "mentions_evidence": False,
        "marks_uncertainty": False,
        "needs_human_review": True,
    }


In [ ]:
sample_answer = "圖片中可見 TOTAL TWD 300；日期可能是 2026-09-16，但仍需人工核對。"
evaluation = evaluate_vision_answer(sample_answer)
assert evaluation["not_empty"] is True
assert evaluation["needs_human_review"] is True
print(evaluation)

## 8. Streamlit 專案銜接

正式配套專案位於 `week12/week12_vision_app/`：

1. `app.py` 專注上傳、預覽、表單與結果顯示。
2. `vision_utils.py` 集中圖片驗證、data URL、prompt、API 與錯誤分流。
3. API 只在送出表單後呼叫，避免 widget rerun 重複產生成本。
4. 圖片結果存在 `st.session_state`；切換畫面時不必重送 API。
5. 收據／表單模式以 structured output 顯示欄位與 JSON。

## 完成檢核與課後任務

- [ ] 能說明 `input_text` + `input_image` 的 content 結構。
- [ ] 能把圖片 bytes 轉成 data URL，且不輸出完整 Base64。
- [ ] 完成上傳驗證、detail 選擇、資料正規化與基本評估。
- [ ] 用虛構收據啟動 App，先測本機驗證，再決定是否開啟付費 API。
- [ ] 比較圖片描述、圖片問答、截圖檢查與結構化抽取的 prompt 差異。
- [ ] 記錄一個 Vision 看錯的案例，說明如何以 UI、prompt 或人工覆核降低風險。

## 官方參考與下一週橋接

- [OpenAI Images and vision](https://developers.openai.com/api/docs/guides/images-vision)
- [OpenAI Responses API](https://developers.openai.com/api/reference/resources/responses/methods/create)
- [GPT-5.4 Mini model](https://developers.openai.com/api/docs/models/gpt-5.4-mini)
- [Streamlit file uploader](https://docs.streamlit.io/develop/api-reference/widgets/st.file_uploader)

第 13 週進入 **Function Calling / Tool Calling 與 Skill-like 模組設計**。本週的 `validate_image()`、`build_task_prompt()` 與 `analyze_image()` 已示範把任務邏輯拆成明確函式；下週會進一步把具備輸入 schema、輸出與錯誤處理的函式提供給模型選擇與呼叫。